# Titanic Survival Modeling: Boosting vs Random Forest

This notebook trains and evaluates **AdaBoost**, **Gradient Boosting**, **XGBoost**, and **Random Forest** on `titanic.csv`, then compares their performance and gives a concise inference.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from xgboost import XGBClassifier

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('titanic.csv')
display(df.head())
print(df.isnull().sum())

In [ ]:
X = df.drop(columns=['Survived', 'Name', 'Ticket', 'Cabin'])
y = df['Survived'].astype(int)

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(exclude=['object']).columns.tolist()

In [ ]:
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [ ]:
models = {
    'AdaBoost': AdaBoostClassifier(n_estimators=200, learning_rate=0.5, random_state=RANDOM_STATE),
    'GradientBoost': GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.9, colsample_bytree=0.9, eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1),
    'RandomForest': RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1)
}

results = []
fitted_models = {}

for name, model in models.items():
    clf = Pipeline([('preprocessor', preprocessor), ('model', model)])
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'ROC_AUC': roc_auc_score(y_test, y_proba)
    })
    fitted_models[name] = clf

results_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
display(results_df)

In [ ]:
best_name = results_df.loc[0, 'Model']
best_model = fitted_models[best_name]

y_pred_best = best_model.predict(X_test)
test_out = X_test.copy()
test_out['Actual_Survived'] = y_test.values
test_out['Predicted_Survived'] = y_pred_best
test_out['Survival_Probability'] = best_model.predict_proba(X_test)[:, 1]

print(f"Best Model: {best_name}")
print(f"Predicted Survivors: {(y_pred_best == 1).sum()} / {len(y_pred_best)}")
display(test_out.head(10))

In [ ]:
print(classification_report(y_test, y_pred_best))

## Inference
Compare Accuracy, F1, and ROC_AUC across all four models in `results_df`. The best-performing model's predictions and classification report are shown in the last two cells.